In [31]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap
import warnings

warnings.filterwarnings('ignore')

In [32]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════
BASE_DIR = r'..\data\features'
OUT_DIR  = r'..\res'
TARGET   = 'nps_weighted_score'
RANDOM_SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

In [33]:
# ═══════════════════════════════════════════════════════════
# 1. LOAD & PREPROCESS DATA
# ═══════════════════════════════════════════════════════════
def load_and_preprocess():
    print("Loading and merging datasets...")
    # Load
    nps   = pd.read_csv(os.path.join(BASE_DIR, 'nps_aggregated_features.csv'))
    txn   = pd.read_csv(os.path.join(BASE_DIR, 'transaction_features.csv'))
    deliv = pd.read_excel(os.path.join(BASE_DIR, 'delivery_features.xlsx'))
    plano = pd.read_csv(os.path.join(BASE_DIR, 'planogram_features.csv'))
    shop  = pd.read_csv(os.path.join(BASE_DIR, 'shop.csv'))
    ms    = pd.read_excel(os.path.join(BASE_DIR, 'ms_features.xlsx'))

    shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')
    deliv = deliv.rename(columns = {
        "cua_hang": "store_id",
        "month":"YearMonth"
    })

    # Merge chain
    merged = txn.merge(nps, on=['store_id', 'YearMonth'], how='inner') \
                .merge(deliv, on=['store_id', 'YearMonth'], how='left') \
                .merge(plano, on=['store_id', 'YearMonth'], how='left') \
                .merge(ms, on=['store_id', 'YearMonth'], how='left')
    
    # Filter Date & Merge Shop
    merged = merged[merged['YearMonth'].between('2025-10', '2026-02')].merge(shop, on='store_id', how='left')

    # Tenure calculation
    period = pd.to_datetime(merged['YearMonth'], format='%Y-%m')
    merged['tenure'] = (period.dt.year - merged['khai_truong_date'].dt.year) * 12 + \
                       (period.dt.month - merged['khai_truong_date'].dt.month)


    # Feature Split
    drop_cols = ['store_id', 'YearMonth', TARGET, 'satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket', 'khai_truong_date', '_period']
    features = [c for c in merged.columns if c not in drop_cols]
    
    X, y = merged[features], merged[TARGET]
    
    # Drop NaNs
    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

    # Drop Collinear Variables (>0.7)
    corr_matrix = X.corr(method='pearson')
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper_tri.columns if any(upper_tri[c].abs() > 0.7)]
    
    X = X.drop(columns=to_drop)
    return X, y, X.columns.tolist()

# ═══════════════════════════════════════════════════════════
# 2. METRICS HELPER
# ═══════════════════════════════════════════════════════════
def get_metrics(y_true, y_pred):
    m = y_true != 0
    mape_score = np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100 if m.sum() > 0 else np.nan
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': mape_score,
        'R2':   r2_score(y_true, y_pred)
    }


In [34]:
# ═══════════════════════════════════════════════════════════
# 3. MAIN PIPELINE
# ═══════════════════════════════════════════════════════════
if __name__ == "__main__":
    X, y, feature_cols = load_and_preprocess()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

    # Model definition & Hyperparameter grid
    models_config = {
        'Baseline (Mean)': (DummyRegressor(strategy='mean'), None),
        
        'Decision Tree': (
            DecisionTreeRegressor(random_state=RANDOM_SEED),
            {'max_depth': [3, 5, 7, None], 'min_samples_split': [5, 10, 20], 'min_samples_leaf': [2, 4, 6], 'ccp_alpha': [0.0, 0.01, 0.05]}
        ),
        
        'Random Forest': (
            RandomForestRegressor(random_state=RANDOM_SEED),
            {'n_estimators': [200, 300, 500], 'max_depth': [5, 10, 15, None], 'min_samples_split': [5, 10], 'min_samples_leaf': [2, 4], 'max_features': ['sqrt', 0.5, 0.7]}
        ),
        
        'XGBoost': (
            XGBRegressor(random_state=RANDOM_SEED, verbosity=0, min_child_weight=5, reg_lambda=1.5, reg_alpha=0.5, colsample_bytree=0.8),
            {'n_estimators': [100, 200, 300], 'max_depth': [2, 3], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7, 0.8]}
        )
    }

    best_models = {}
    eval_results = []

    print("\nTraining models...")
    for name, (model, params) in models_config.items():
        print(f"  -> {name}")
        if params:
            search = RandomizedSearchCV(model, params, n_iter=30, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, random_state=RANDOM_SEED)
            search.fit(X_train, y_train)
            best_model = search.best_estimator_
            print(f"     Best params: {search.best_params_}")
        else:
            best_model = model.fit(X_train, y_train) # For DummyRegressor
            
        best_models[name] = best_model
        
        # Calculate Metrics
        train_m = get_metrics(np.array(y_train), best_model.predict(X_train))
        test_m  = get_metrics(np.array(y_test), best_model.predict(X_test))
        eval_results.append({'Model': name, **{f'Train {k}': v for k, v in train_m.items()}, **{f'Test {k}': v for k, v in test_m.items()}})

    # Save Evaluation Table
    results_df = pd.DataFrame(eval_results).set_index('Model')
    print("\n=== EVALUATION RESULTS ===")
    print(results_df.round(4))
    results_df.to_csv(os.path.join(OUT_DIR, 'model_results.csv'))


Loading and merging datasets...

Training models...
  -> Baseline (Mean)
  -> Decision Tree
     Best params: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 5, 'ccp_alpha': 0.01}
  -> Random Forest
     Best params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': 5}
  -> XGBoost
     Best params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05}

=== EVALUATION RESULTS ===
                 Train MAE  Train RMSE  Train MAPE  Train R2  Test MAE  \
Model                                                                    
Baseline (Mean)     1.1279      1.4789     63.2327    0.0000    1.1500   
Decision Tree       0.0985      0.3281      6.7268    0.9508    0.3409   
Random Forest       0.2125      0.4707     13.7106    0.8987    0.3911   
XGBoost             0.1682      0.3544     10.9316    0.9426    0.3131   

                 Test RMSE  Test MAPE  Test R2  
Model                          

In [35]:
# ═══════════════════════════════════════════════════════════
# 4. PLOTS & SHAP VALUES
# ═══════════════════════════════════════════════════════════
print("\nPlotting Feature Importances and SHAP...")
tree_models = {k: v for k, v in best_models.items() if k != 'Baseline (Mean)'}
colors = ['#4C72B0', '#55A868', '#C44E52']

# ── 4.1 Built-in Feature Importance ──
fig, axes = plt.subplots(1, len(tree_models), figsize=(22, 7))
for ax, (name, model), color in zip(axes, tree_models.items(), colors):
    imps = model.feature_importances_
    idx  = np.argsort(imps)
    ax.barh(np.array(feature_cols)[idx], imps[idx], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'feature_importance_builtin.png'), dpi=150, bbox_inches='tight')
plt.close()

# ── 4.2 SHAP Values ──
shap_df = pd.DataFrame({'Feature': feature_cols})
X_test_r = X_test.reset_index(drop=True)

for name, model in tree_models.items():
    try:
        # Beeswarm Plot
        shap_expl = shap.Explainer(model, X_train)(X_test_r)
        plt.figure(figsize=(11, 7))
        shap.plots.beeswarm(shap_expl, show=False, max_display=len(feature_cols))
        plt.title(f'SHAP Beeswarm — {name}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"shap_beeswarm_{name.replace(' ', '_').lower()}.png"), dpi=150, bbox_inches='tight')
        plt.close()

        # Importance Table values
        shap_vals = shap.TreeExplainer(model).shap_values(X_test_r)
        if len(shap_vals.shape) == 2:
            shap_df[f'SHAP_{name}'] = np.abs(shap_vals).mean(axis=0)
            
    except Exception as e:
        print(f"Error computing SHAP values for {name}: {e}")

if 'SHAP_XGBoost' in shap_df.columns:
    shap_df = shap_df.sort_values('SHAP_XGBoost', ascending=False)

shap_df.to_csv(os.path.join(OUT_DIR, 'shap_importance_table.csv'), index=False)
print(f"\nPipeline Complete. All outputs saved to {OUT_DIR}")


Plotting Feature Importances and SHAP...


Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.



Pipeline Complete. All outputs saved to ..\res
